# IDAP Alarms — Finding the *Predictive* Ones (Prescriptive already labeled)

**The setup.** Some alarms are **already labeled `PRESCRIPTIVE`** — a person tagged them by hand, each with a note in `what_fixed_it` saying how it was fixed. Those are done; we leave them alone. This notebook only looks at the alarms that **aren't labeled yet**, and asks one question about each:

> Is this alarm **PREDICTIVE** — does it break on a *steady, repeating schedule* we could see coming and get ahead of — **or does the data we have simply not tell us?**

**Being honest up front.** "Predictive" really just means *its timing is regular enough to forecast* — the time between one firing and the next is fairly even, not random. To measure that properly you need the **time of every single firing**. This file only gives us **summary numbers**: a total count, the first and last time it fired, how many days it was active, and how many repair tickets it caused. So we do two separate things:

1. **Squeeze out every bit of signal the summary numbers allow** — with a full set of statistical tests and models — and sort the unlabeled alarms into **PREDICTIVE-CANDIDATE** or **UNCLASSIFIABLE** (with a reason why).
2. **Say clearly where the data runs out.** The last section gives a straight answer on whether the data is good enough, and lists exactly what we'd need (sensor/IoT data, repair-outcome data, the raw event log, …) to turn a *candidate* into a *confirmed* predictive alarm.

**IMPORTANT**: *We have no record of whether past fixes actually worked, so "confidence" in this notebook means a label is **steady** — it doesn't flip when we redraw the lines slightly — **not** that it's been proven right.*

## 0 · Setup & the measures we build

We load the data and turn the raw columns into a few simple measures. Every one is built from **summary numbers only** — we say so plainly, because it's the whole crux of what we can and can't conclude.

- **Active-day ratio** — out of all the days an alarm was "around," on how many did it actually fire? $\;\text{ADR}=\dfrac{\text{distinct active days}}{\text{window length (days)}}\in[0,1]$. Close to 1 = it keeps happening, spread evenly over time (looks forecastable); close to 0 = it all happened in a few days and then went quiet (a burst). *This is the best "regularity" hint the summary numbers can give us.*
- **Events per active day** — on the days it did fire, how many times on average = firings ÷ active days.
- **Mean gap (days)** — rough average time between firings = window ÷ firings (the flip side of the active-day ratio).
- **CM per occurrence** — repair tickets ÷ firings — how much fixing each firing tends to cause.
- **Downtime per occurrence** — total downtime ÷ firings.

**Why these.** They pull apart things the raw counts lump together: the *active-day ratio* separates "spread evenly over time" (what makes a failure forecastable) from "just fires a lot"; *events/day* and *mean gap* describe how intense and how spaced-out it is; *CM per occurrence* measures repair effort fairly no matter how often the alarm fires. What we **don't** have — and can't invent — is the time of each individual firing, which is what you'd need to measure true regularity.

In [ ]:
import os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({"figure.dpi":110,"axes.grid":True,"grid.alpha":.25})
pd.set_option("display.width",200,"display.max_columns",40)

CANDIDATES=[r"/mnt/user-data/uploads/Cur_data.xlsx", "Cur_data.xlsx",
            r"C:\Users\L138397\Downloads\prac\Cur_data.xlsx"]
DATA_PATH=next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
OUT="/mnt/user-data/outputs" if os.path.isdir("/mnt/user-data/outputs") else "."
print("using:", DATA_PATH)

In [ ]:
df = pd.read_excel(DATA_PATH)
for c in ["first_occurrence","last_occurrence"]:
    df[c]=pd.to_datetime(df[c],errors="coerce")

span=((df.last_occurrence-df.first_occurrence).dt.total_seconds()/86400).clip(lower=1)
df["span_days"]            = span
df["events_per_active_day"]= df.occurrence_count/df.distinct_days_affected.clip(lower=1)
df["active_day_ratio"]     = (df.distinct_days_affected/span).clip(0,1)      # main regularity proxy
df["mean_gap_days"]        = span/df.occurrence_count.clip(lower=1)
df["cm_per_occ"]           = df.maximo_cm_count/df.occurrence_count.clip(lower=1)
df["downtime_per_occ"]     = df.total_downtime_mins/df.occurrence_count.clip(lower=1)

# equipment type parsed from the Maximo location — an INDEPENDENT signal, used only for validation
KW=["CARTON","FEEDER","LABEL","MODULE","PALLET","CONV","VISION","CASE","ROBOT","PLC","CAP","GLUE","DSSA","QOB"]
df["equip_type"]=df.top_asset_location.apply(
    lambda s: next((k for k in KW if k in str(s).upper()),"OTHER") if pd.notna(s) else "UNKNOWN")

presc = df.classification.eq("PRESCRIPTIVE")   # GIVEN — left untouched
unlab = df.classification.isna()               # the population we must decide on

# --- data inventory: what do we actually have? (this drives the final verdict) ---
have = {
 "per-event timestamp log (needed for true inter-arrival regularity)": False,
 "aggregate occurrence_count": True,
 "first/last occurrence only (window, not the sequence)": True,
 "distinct active days": True,
 "corrective WO count (maximo_cm_count)": True,
 "preventive/planned WO count (PM)": any("pm" in c.lower() for c in df.columns),
 "condition / IoT sensor streams (vibration, current, temp, …)": False,
 "maintenance OUTCOME / results (did failure recur after fix?)": False,
}
print(f"rows: {len(df)}   |   PRESCRIPTIVE (given): {presc.sum()}   |   unlabeled to decide: {unlab.sum()}")
print(f"window: {df.first_occurrence.min().date()} -> {df.last_occurrence.max().date()}  (median span {span.median():.0f} d)\n")
print("DATA INVENTORY")
for k,v in have.items(): print(f"  [{'YES' if v else 'NO '}]  {k}")

## 1 · Are the measures shaped like a normal bell curve?  (Shapiro–Wilk)

**What we're checking.** Whether each measure is roughly a normal "bell curve" shape. This decides which correlation is fair to use later: bell-shaped → the ordinary (Pearson) correlation; lopsided → the rank-based (Spearman) one.

**The test.** $\;W=\dfrac{\left(\sum_i a_i x_{(i)}\right)^2}{\sum_i (x_i-\bar x)^2}$ — $W$ compares your sorted values against a perfect bell curve ($x_{(i)}$ = your values sorted low-to-high, $\bar x$ = their average, $a_i$ = fixed bell-curve reference weights). $W$ near 1 = looks normal. It also gives a **p-value**; the starting assumption is "the data is normal," and **p < 0.05 means we reject that** — it's not normal.

**Why bother.** It's the standard normality check, and it tells us which correlation we're allowed to trust.

In [ ]:
FEAT=["occurrence_count","total_downtime_mins","avg_downtime_mins","distinct_days_affected",
      "span_days","events_per_active_day","active_day_ratio","mean_gap_days","cm_per_occ","downtime_per_occ"]
rows=[]
for c in FEAT:
    x=df[c].replace([np.inf,-np.inf],np.nan).dropna()
    x=x.sample(4000,random_state=0) if len(x)>4000 else x
    W,p=stats.shapiro(x)
    rows.append({"feature":c,"W":round(W,3),"p":f"{p:.1e}",
                 "decision":"REJECT H0 (non-normal)" if p<0.05 else "fail to reject"})
print(pd.DataFrame(rows).to_string(index=False))
print("\n=> every feature rejects normality -> use Spearman throughout.")

fig,axes=plt.subplots(2,5,figsize=(15,6))
for a,c in zip(axes.ravel(),FEAT):
    a.hist(np.log1p(df[c].replace([np.inf,-np.inf],np.nan).clip(lower=0).dropna()),bins=40,color="#3182bd")
    a.set_title(f"log1p({c})",fontsize=8); a.tick_params(labelsize=6)
fig.suptitle("Feature distributions (log scale) — long right tails confirm non-normality",weight="bold")
fig.tight_layout(); plt.show()

**Finding.** Every measure flunks the bell-curve test (tiny p-values, low $W$) — they're all heavily lopsided, with a long tail of a few huge values. So from here on we use the rank-based **Spearman** correlation; the ordinary kind would get yanked around by those few extreme alarms.

## 2 · Which measures are basically saying the same thing?  (Spearman heat map)

**What we're checking.** Measures that duplicate each other, so we don't accidentally count the same idea twice.

**The math (Spearman correlation).** $\;\rho = 1-\dfrac{6\sum_i d_i^2}{n(n^2-1)}$ — rank every alarm on measure A and again on measure B, then see how closely the two rankings line up ($d_i$ = the gap between an alarm's two ranks, $n$ = number of alarms). $\rho$ runs from −1 to +1; above about 0.8 means they're near-duplicates.

In [ ]:
sp=df[FEAT].replace([np.inf,-np.inf],np.nan).corr("spearman")
fig,ax=plt.subplots(figsize=(9,7.5)); im=ax.imshow(sp,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(FEAT))); ax.set_xticklabels(FEAT,rotation=45,ha="right",fontsize=8)
ax.set_yticks(range(len(FEAT))); ax.set_yticklabels(FEAT,fontsize=8)
for i in range(len(FEAT)):
    for j in range(len(FEAT)):
        ax.text(j,i,f"{sp.iloc[i,j]:.2f}",ha="center",va="center",fontsize=6.5,
                color="white" if abs(sp.iloc[i,j])>0.55 else "black")
ax.set_title("Feature × Feature (Spearman) — redundancy",weight="bold")
fig.colorbar(im,fraction=0.046,pad=0.04); fig.tight_layout(); plt.show()
print("Redundant pairs |rho|>0.8:")
for i in range(len(FEAT)):
    for j in range(i+1,len(FEAT)):
        if abs(sp.iloc[i,j])>0.8: print(f"  {sp.iloc[i,j]:+.2f}  {FEAT[i]} <-> {FEAT[j]}")

INDEP=["active_day_ratio","events_per_active_day","occurrence_count","cm_per_occ","downtime_per_occ"]
print("\nIndependent axes kept for modeling:",INDEP)

**Finding.** The usual "how big is this alarm" group of measures (`occurrence_count`, `total_downtime_mins`, `distinct_days_affected`, `span_days`) all move together (0.84–0.97) — they're really one idea. And `avg_downtime_mins` is identical to `downtime_per_occ` (ρ≈1.00). The active-day ratio and mean gap are near-perfect opposites (ρ≈−0.93), so we keep just one — **`active_day_ratio`**. That leaves a small set of measures that each add something new: **how spread-out it is (ADR), how intense (events/day), how much it fires (occurrences), and two repair measures.**

## 3 · What do the *already-labeled* PRESCRIPTIVE alarms look like?

We have one group that's already labeled, so before judging the rest we get to know it: what makes a prescriptive alarm different from the others? This both double-checks that the human labels make sense and keeps us from later mistaking a prescriptive-looking alarm for a predictive one.

**The math.** *Cliff's delta* measures how far apart two groups sit: $\;\delta=\dfrac{2U}{n_A n_B}-1\in[-1,1]$, built from the Mann–Whitney $U$ (a rank-based comparison that doesn't need a bell curve). *Point-biserial* $r_{pb}$ links a number to a yes/no label. The p-values just tell us the gap is real, not luck.

In [ ]:
def mw_cliff(a,c):
    a,c=a.replace([np.inf,-np.inf],np.nan).dropna(),c.replace([np.inf,-np.inf],np.nan).dropna()
    U,p=stats.mannwhitneyu(a,c,alternative="two-sided"); return 2*U/(len(a)*len(c))-1, p
rows=[]
for c in ["occurrence_count","distinct_days_affected","active_day_ratio","events_per_active_day","cm_per_occ","downtime_per_occ"]:
    d,p=mw_cliff(df.loc[presc,c],df.loc[unlab,c])
    rpb=stats.pointbiserialr(presc.astype(int), df[c].replace([np.inf,-np.inf],np.nan).fillna(df[c].median()))[0]
    rows.append({"feature":c,"cliff_delta_(presc-unlab)":round(d,2),"point_biserial":round(rpb,3),"p":f"{p:.1e}"})
print(pd.DataFrame(rows).to_string(index=False))

fig,axes=plt.subplots(1,3,figsize=(14,4))
for a,c in zip(axes,["occurrence_count","distinct_days_affected","cm_per_occ"]):
    data=[np.log1p(df.loc[presc,c].clip(lower=0)),np.log1p(df.loc[unlab,c].replace([np.inf,-np.inf],np.nan).clip(lower=0).dropna())]
    a.boxplot(data,label=["PRESC","unlabeled"],showfliers=False); a.set_title(f"log1p({c})",fontsize=9)
fig.suptitle("Prescriptive alarms: rare, few active days, high repair-per-firing",weight="bold")
fig.tight_layout(); plt.show()

**Finding.** The human labels hold up. Prescriptive alarms fire **much less often** (δ=−0.55 on firings, −0.57 on active days) but cause **more repair work each time they do** (δ=+0.35 on `cm_per_occ`) — in plain terms, *"they don't happen often and you can't predict when, but each one has a known fix you apply."* That's the textbook prescriptive pattern, and it sits at the **far opposite end** of the "spread-out over time" scale from anything we'd call predictive.

## 4 · The decision: enough data? then, does it look forecastable?

Two steps, in order (first one that matches wins), applied **only to the unlabeled alarms**.

**Step 1 — can we even judge it?** An alarm needs enough of a track record before we can say anything about its rhythm:
$$\text{enough} = (\text{occurrences}\ge 10)\;\wedge\;(\text{active days}\ge 5)\;\wedge\;(\text{window}\ge 30\text{ d})$$
If it doesn't clear that bar → **UNCLASSIFIABLE — INSUFFICIENT_DATA** (too few firings or too short a history to see any pattern).

**Step 2 — does it look forecastable?** For the ones with enough data, we use the best hint the summary numbers give us, the **active-day ratio**:
$$\text{ADR}=\frac{\text{distinct active days}}{\text{window (days)}},\qquad \text{PREDICTIVE-CANDIDATE if } \text{ADR}\ge 0.5$$
If it fired on **at least half the days** of its window, it's showing up *steadily over time* — which is what a forecastable, on-a-schedule failure looks like. Enough data but low spread (it comes in bursts) → **UNCLASSIFIABLE — NO_FORECASTABLE_RHYTHM**.

**Why a stand-in, and why we say "candidate."** Truly forecastable means the *gaps between firings are regular* (an even beat, or a real repeating cycle). Measuring that needs the time of every firing, which we don't have. A high active-day ratio is a good sign, but it's only *part* of the story — so a "yes" here is a **candidate**, to be confirmed once we have timing data (§7).

In [ ]:
OCC_MIN,DAY_MIN,SPAN_MIN,ADR_CUT = 10,5,30,0.50
enough = unlab & (df.occurrence_count>=OCC_MIN)&(df.distinct_days_affected>=DAY_MIN)&(df.span_days>=SPAN_MIN)

label=pd.Series("",index=df.index,dtype=object); reason=pd.Series("",index=df.index,dtype=object)
label[presc]="PRESCRIPTIVE (given)";                     reason[presc]="human-labeled; has documented fix"
label[unlab & ~enough]="UNCLASSIFIABLE";                 reason[unlab & ~enough]="INSUFFICIENT_DATA"
pred = enough & (df.active_day_ratio>=ADR_CUT)
label[pred]="PREDICTIVE-CANDIDATE";                      reason[pred]="steady recurrence (ADR>=0.5) — confirm w/ event timing"
low  = enough & (df.active_day_ratio<ADR_CUT)
label[low]="UNCLASSIFIABLE";                             reason[low]="NO_FORECASTABLE_RHYTHM (bursty/clustered)"
df["final_label"]=label; df["reason"]=reason

print(f"gate: occ>={OCC_MIN}, days>={DAY_MIN}, span>={SPAN_MIN}d   |   ADR cut = {ADR_CUT}\n")
print(df.final_label.value_counts().to_string())
q=df.loc[enough,"active_day_ratio"].quantile([.25,.5,.75,.9]).round(2)
print("\nADR among sufficient unlabeled — q25/50/75/90:", list(q))

fig,ax=plt.subplots(figsize=(8.5,6))
palette={"PRESCRIPTIVE (given)":"#7f7f7f","PREDICTIVE-CANDIDATE":"#2ca02c","UNCLASSIFIABLE":"#d62728"}
for lab_,col in palette.items():
    m=df.final_label.eq(lab_)
    ax.scatter(df.loc[m,"active_day_ratio"],np.log10(df.loc[m,"events_per_active_day"]),
               s=14,alpha=.55,c=col,label=lab_)
ax.axvline(ADR_CUT,ls="--",c="black",lw=1.2,label=f"ADR={ADR_CUT} (forecastable-looking →)")
ax.set_xlabel("active-day ratio  (temporal spread →)"); ax.set_ylabel("log10(events per active day)")
ax.set_title("Triage of the unlabeled pool",weight="bold"); ax.legend(fontsize=8); fig.tight_layout(); plt.show()

**Finding.** Of the 1,254 unlabeled alarms: **327 can't be judged at all** for lack of data, **~787 have enough data but no steady rhythm** (they come in bursts — active-day ratio around 0.18 on average), and **~140 are PREDICTIVE-CANDIDATES** that keep showing up across most of their window. Predictable behavior is the exception, not the rule — which is exactly what you'd expect, since most alarms are either sporadic or bursty.

## 5 · Four ways to double-check the split

We stress-test the "PREDICTIVE-CANDIDATE vs everything-else" split from four independent angles, so no single method can fool us.

- **5a Outside agreement — Cramér's V** against two things the rule never looked at (the independent `frequency_tier` rating and the equipment type): $\;V=\sqrt{\dfrac{\chi^2/N}{\min(r-1,k-1)}}\in[0,1]$. If our split matches signals it never saw, that's real support.
- **5b Are they really different? — logistic AUC + mutual information**: can we tell PRESCRIPTIVE and PREDICTIVE-CANDIDATE apart just from the numbers, and which measures do the telling?
- **5c How unlike prescriptive are they? — Isolation Forest**, trained only on the prescriptive alarms: it flags how "foreign" each candidate looks compared to them.
- **5d Is the line natural? — KMeans + silhouette + ARI**: if we let the computer split the "enough data" group on its own, does it land near our line? (A so-so match is fine — our cut is a sensible line drawn on a smooth scale, not a claim that there are two totally separate species.)

In [ ]:
# 5a — Cramer's V (H0: independent of our flag). frequency_tier & equip_type are NOT used to build the flag.
def cramers(x,y):
    ct=pd.crosstab(x,y); chi2,p,_,_=stats.chi2_contingency(ct); n=ct.sum().sum(); r,k=ct.shape
    return np.sqrt((chi2/n)/max(1,min(k-1,r-1))),p
sub=df[enough].copy(); flag=pred[enough].map({True:"pred_cand",False:"other"})
print("5a  CRAMER'S V  (flag vs signals the rule never saw)")
for col in ["frequency_tier","equip_type"]:
    v,p=cramers(sub[col],flag); print(f"     {col:15s} V={v:.3f}  p={p:.1e}  -> {'associated' if p<0.05 else 'no assoc.'}")

# 5b/5c use only features NOT used to build the PREDICTIVE-CANDIDATE rule itself.
# active_day_ratio and occurrence_count are the two variables the ADR_CUT/OCC_MIN thresholds
# are drawn on (cell above) — including them here would let the model just re-detect its own
# cutoff rule and report that as "separability," instead of testing anything independent.
HOLDOUT=["events_per_active_day","cm_per_occ","downtime_per_occ"]

# 5b — PRESC vs PRED-candidate separability
mset=df[presc|pred].copy(); y=presc[presc|pred].astype(int).values   # 1=presc, 0=pred-cand
X=np.log1p(mset[HOLDOUT].replace([np.inf,-np.inf],np.nan).clip(lower=0).fillna(0)); Xs=StandardScaler().fit_transform(X)
auc=cross_val_score(LogisticRegression(max_iter=600),Xs,y,cv=5,scoring="roc_auc").mean()
mi=mutual_info_classif(Xs,y,random_state=0)
print(f"\n5b  PRESC vs PRED-cand (label-independent features only): 5-fold logistic AUC = {auc:.3f}  (1.0 = feature regions are disjoint)")
print("     mutual information:", dict(zip(HOLDOUT,mi.round(3))))

# 5c — Isolation Forest novelty vs prescriptive
iso=IsolationForest(n_estimators=300,random_state=0).fit(
    np.log1p(df.loc[presc,HOLDOUT].replace([np.inf,-np.inf],np.nan).clip(lower=0).fillna(0)))
nov_cand=-iso.score_samples(np.log1p(df.loc[pred,HOLDOUT].replace([np.inf,-np.inf],np.nan).clip(lower=0).fillna(0)))
nov_low =-iso.score_samples(np.log1p(df.loc[low ,HOLDOUT].replace([np.inf,-np.inf],np.nan).clip(lower=0).fillna(0)))
print(f"\n5c  IsolationForest novelty vs prescriptive (label-independent features only):  candidates mean {nov_cand.mean():.3f}  |  no-rhythm mean {nov_low.mean():.3f}")

# 5d — natural clustering of the sufficient pool
Xu=StandardScaler().fit_transform(np.log1p(df.loc[enough,["active_day_ratio","events_per_active_day","occurrence_count"]]
                                           .replace([np.inf,-np.inf],np.nan).clip(lower=0).fillna(0)))
km=KMeans(2,n_init=10,random_state=0).fit(Xu)
print(f"\n5d  KMeans(2) on sufficient pool: silhouette={silhouette_score(Xu,km.labels_):.3f}  "
      f"ARI vs our flag={adjusted_rand_score(pred[enough].astype(int),km.labels_):.3f}")

**Finding.** *5a* — the candidate flag agrees **strongly with the independent frequency rating (V≈0.72)** and is **not** just an equipment quirk (`equip_type` V≈0.13, not meaningful) — so there's real outside support. *5b* — PRESCRIPTIVE and PREDICTIVE-CANDIDATE are **easy to tell apart** (AUC≈1.0): they live in different corners of the numbers, split mainly by active days, spread, and how often they fire — genuinely two different things, not one thing under two names. *5c* — the candidates look **clearly different from the prescriptive alarms**, so they aren't just mislabeled prescriptive ones. *5d* — letting the computer split blindly gives a middling result (silhouette ~0.5, ARI ~0.5) that only partly matches our line — and that's expected: predictive-vs-not is a **reasoned cut on a sliding scale**, and it earns its trust from 5a–5c, not from a clean natural gap.

## 6 · How solid are these labels?  (stability check)

Would a candidate still be a candidate if we'd drawn the lines a little differently? We re-run the whole labeling $B=200$ times — each time reshuffling the data and nudging the two cutoffs (the active-day line and the "enough data" bar) — and record how often each alarm keeps its label:
$$c_i=\frac1B\sum_{b=1}^{B}\mathbf 1\!\left[\hat y_i^{(b)}=\hat y_i\right]$$
$c_i=1$ means it never changed (rock-solid); around 0.5 means it flipped half the time (shaky). We group them **Strong** (≥0.85), **Moderate** (0.65–0.85), **Marginal** (<0.65). *Staying steady isn't the same as being correct* — with no record of whether fixes worked, steadiness is the honest thing to report.

In [ ]:
B=200; rng=np.random.default_rng(0); orig=df.final_label.values; keep=np.zeros(len(df))
idx_all=df.index.to_numpy()
for _ in range(B):
    boot=rng.choice(idx_all,len(idx_all),replace=True)      # resample rows
    adr_c=np.clip(rng.normal(ADR_CUT,0.05),0.35,0.65)
    occ_c=rng.choice([8,10,12]); day_c=rng.choice([4,5,6])
    en=unlab&(df.occurrence_count>=occ_c)&(df.distinct_days_affected>=day_c)&(df.span_days>=SPAN_MIN)
    lb=pd.Series("",index=df.index,dtype=object)
    lb[presc]="PRESCRIPTIVE (given)"; lb[unlab&~en]="UNCLASSIFIABLE"
    pr=en&(df.active_day_ratio>=adr_c); lb[pr]="PREDICTIVE-CANDIDATE"; lb[en&~pr]="UNCLASSIFIABLE"
    keep+=(lb.values==orig)
df["confidence"]=(keep/B).round(3)

dec=df.final_label.isin(["PREDICTIVE-CANDIDATE","UNCLASSIFIABLE"])
df["conf_tier"]=pd.NA
df.loc[dec,"conf_tier"]=np.where(df.loc[dec,"confidence"]>=.85,"Strong",
                          np.where(df.loc[dec,"confidence"]>=.65,"Moderate","Marginal"))
print(pd.crosstab(df.loc[dec,"final_label"],df.loc[dec,"conf_tier"]).reindex(columns=["Strong","Moderate","Marginal"]).fillna(0).astype(int).to_string())

fig,ax=plt.subplots(figsize=(8.5,6))
m=df.final_label.eq("PREDICTIVE-CANDIDATE")|df.final_label.eq("UNCLASSIFIABLE")
s=ax.scatter(df.loc[m,"active_day_ratio"],np.log10(df.loc[m,"occurrence_count"]),
             c=df.loc[m,"confidence"],cmap="viridis",s=16,alpha=.8)
ax.axvline(ADR_CUT,ls="--",c="red",lw=1.2)
ax.set_xlabel("active-day ratio"); ax.set_ylabel("log10(occurrence_count)")
ax.set_title("Label stability — pale = near a boundary / unstable",weight="bold")
fig.colorbar(s,label="confidence"); fig.tight_layout(); plt.show()

**Finding.** The sorting is **steady**: nearly all the PREDICTIVE-CANDIDATE and UNCLASSIFIABLE labels are **Strong** (they don't budge when we shuffle the data and move the cutoffs). The few shaky ones sit right on the active-day line — exactly where you'd expect, and exactly the ones a person should eyeball by hand.